In [17]:
import torch
from torchvision import datasets, transforms

In [18]:
transform = transforms.ToTensor()

In [19]:
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

In [20]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True #DMA from RAM to GPU without paging
)

test_loader = DataLoader(
    test_data, 
    batch_size=64, 
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [21]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3),  # [1,28,28] → [8,26,26]  
    # 28 -> 26 =>  Output = (Input - Kernel + 2 × Padding) / Stride + 1
    nn.ReLU(),  # negative values = 0
    nn.MaxPool2d(2),    # [8,26,26] → [8,13,13]
    nn.Flatten(),    # [8,13,13] → [1352]
    nn.Linear(8 * 13 * 13, 10)  # 10 digit classes
)

In [22]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
print(device)

cuda


In [23]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for images, labels in train_loader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        epoch_loss += loss.item()
        
    epoch_loss /= len(train_loader)
    print(f"Epoch: {epoch} | Loss: {epoch_loss:.4f}")
        

Epoch: 0 | Loss: 0.3549
Epoch: 1 | Loss: 0.1376
Epoch: 2 | Loss: 0.1031
Epoch: 3 | Loss: 0.0860
Epoch: 4 | Loss: 0.0756


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        predicted = preds.argmax(dim=1)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
accuracy = correct / total
print(accuracy)
        

0.97975
